In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
#from utils import *
#from utils_cpdir import *
from utils_mvcp import *
from utils import *
from utils_spec import *
import os
from tqdm import tqdm
import json
import os
from datetime import date

In [2]:
# Get the current date
todays_date = date.today()
year = str(todays_date.year)
if todays_date.month < 10:
    month = '0'+str(todays_date.month)
else:
    month = str(todays_date.month)

if todays_date.day < 10:
    day = '0'+str(todays_date.day)
else:
    day = str(todays_date.day)

current_date = year+month+day
directory_name = 'results/'+current_date
if not os.path.isdir(directory_name):
    os.mkdir(directory_name)
    
directory_name

'results/20251110'

In [3]:
#directory_name = 'results/20251026'

In [4]:
remove_slopp = pd.read_excel('RochmanSpectra_ToRemove.xlsx', sheet_name="SLOPPe Remove")
mIRageLibraySpectra_Removed = pd.read_csv('../gloves_contamination/mIRageLibraySpectra_Removed.csv')
mIRageLibraySpectra_Removed.iloc[-1,2] = 'PE_15'

dict_slopp_reb = {'PE':'Polyethylene', 
                  'PP':'Polypropylene', 
                  'PSL':'Polystyrene'}

slopp_df = pd.read_csv('processed_data/slopp_data_raman_final_specy.csv')
del slopp_df['Unnamed: 0']
slopp_df.columns = list(np.float64(slopp_df.columns[:-4])) + list(slopp_df.columns[-4:])

slopp_df = slopp_df.query('polymer != "Cotton" and polymer !="Dyed Cellulose"').reset_index(drop=True)
slopp_df.loc[slopp_df['polymer'] == "Polyester",'polymer'] = "Polyethylene Terephthalate"
slopp_df = slopp_df.loc[[x not in np.array(remove_slopp['Example']) for x in slopp_df['id']], :].reset_index(drop=True)

rebecca_raman_df = pd.read_csv('processed_data/rebecca_data_raman_final_specy.csv')
del rebecca_raman_df['Unnamed: 0']
rebecca_raman_df.columns = list(np.float64(rebecca_raman_df.columns[:-4])) + list(rebecca_raman_df.columns[-4:])

rebecca_raman_df = rebecca_raman_df.drop(np.where([x in list(mIRageLibraySpectra_Removed.query('Signal == "Raman"')['ID']) for x in rebecca_raman_df['id']])[0],
                             axis=0).reset_index(drop=True)

rebecca_raman_df = rebecca_raman_df.query('polymer in @dict_slopp_reb').reset_index(drop=True)
rebecca_raman_df['polymer'] = [dict_slopp_reb[x] for x in rebecca_raman_df['polymer']]

In [5]:
folders = np.array(os.listdir('processed_data'))
file_names = folders[['active' in x and 'raman_saturation' in x for x in folders]]
file_names = ['processed_data/'+x for x in file_names]
ids_sat = []
for f in file_names:
    with open(f, 'r') as f:
        saturated_wn_dict = json.load(f)

    ids_sat = ids_sat + list(saturated_wn_dict.keys())
    #ids_sat_list.append(ids_sat)
    

In [6]:
folders = np.array(os.listdir())
file_names = folders[['active_data' in x and 'raman_specy_particle' in x for x in folders]]
file_names

array([], dtype='<U48')

In [7]:
folders = np.array(os.listdir('processed_data_old'))
file_names = folders[['active_data' in x and 'raman_specy_particle' in x for x in folders]]
file_names = ['processed_data_old/'+x for x in file_names]
particle_info_list = []

for f in file_names:
    particle_info = pd.read_csv(f)
    particle_info= particle_info.loc[:,['id','size','area','perimeter','circularity','distance']]
    #del active_data_raman['Unnamed: 0']
    #active_data_raman.columns = list(np.float64(active_data_raman.columns[:-7])) + list(active_data_raman.columns[-7:])

    #active_data_raman['polymer'] = ''
    #active_data_raman['label'] = ''

    particle_info_list.append(particle_info)
    
particle_info = pd.concat(particle_info_list, axis=0).reset_index(drop=True)
particle_info#.to_csv('particle_info_active_data.csv')

,id,size,area,perimeter,circularity,distance
0,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,2.387170,4.554399,8.426622,0.805787,0.347270
1,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,2.920689,6.603032,11.091790,0.674274,0.232162
2,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,2.390522,4.571330,9.162588,0.684073,0.381362
3,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,1.626374,2.133287,6.326505,0.669604,0.190898
4,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,2.835288,6.179760,11.504420,0.586596,0.358090
...,...,...,...,...,...,...
3316,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_130,0.971786,0.732929,3.272318,0.859788,0.182585
3317,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_131,1.090201,0.957468,3.770599,0.845950,0.371000
3318,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_132,0.872364,0.591004,3.126375,0.759536,0.332471
3319,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_133,0.902293,0.603713,2.996222,0.844741,0.252571


In [8]:
folders = np.array(os.listdir('processed_data'))
file_names = folders[['active_data' in x and 'raman_specy.' in x for x in folders]]
file_names = ['processed_data/'+x for x in file_names]
active_data_raman_list = []

for f in file_names:
    active_data_raman = pd.read_csv(f)
    del active_data_raman['Unnamed: 0']
    active_data_raman.columns = list(np.float64(active_data_raman.columns[:-7])) + list(active_data_raman.columns[-7:])

    active_data_raman['polymer'] = ''
    active_data_raman['label'] = ''

    active_data_raman_list.append(active_data_raman)
    
active_data_raman = pd.concat(active_data_raman_list, axis=0).reset_index(drop=True)
ids_no_signal = np.array(active_data_raman.query('no_signal>0')['id'])
del active_data_raman['wn_saturated']
del active_data_raman['no_signal']
del active_data_raman['id_file']

wn_raman = np.float64(slopp_df.columns[:-4])

min_raman_1 = 600
max_raman_1 = 1000

min_raman_2 = 1600
max_raman_2 = 3500

idx_1 = list(np.logical_or(
    np.logical_and(
        wn_raman > min_raman_1,
        wn_raman < max_raman_1),
    np.logical_and(
        wn_raman > min_raman_2,
        wn_raman < max_raman_2))) + [True]*4

min_raman_1 = 600
max_raman_1 = 910

min_raman_2 = 1040
max_raman_2 = 2310

min_raman_3 = 2500
max_raman_3 = 3500

idx_1 = list(
    np.logical_or( 
        np.logical_or(
            np.logical_and(
                wn_raman > min_raman_1,
                wn_raman < max_raman_1),
            np.logical_and(
                wn_raman > min_raman_2,
                wn_raman < max_raman_2)
            ),   
        np.logical_and(
            wn_raman > min_raman_3,
            wn_raman < max_raman_3)
    )
    ) + [True]*4

idx_2 = idx_1[:-4] + [False]*4

slopp_df = slopp_df.loc[:,idx_1]
rebecca_raman_df = rebecca_raman_df.loc[:,idx_1]
active_data_raman = active_data_raman.loc[:,idx_1+[True]]

slopp_df.iloc[:,:-4] = (slopp_df.iloc[:,:-4] - slopp_df.iloc[:,:-4].mean(1).to_numpy()[:,None]) / slopp_df.iloc[:,:-4].std(1).to_numpy()[:,None]
rebecca_raman_df.iloc[:,:-4] = (rebecca_raman_df.iloc[:,:-4] - rebecca_raman_df.iloc[:,:-4].mean(1).to_numpy()[:,None]) / rebecca_raman_df.iloc[:,:-4].std(1).to_numpy()[:,None]
active_data_raman.iloc[:,:-5] = (active_data_raman.iloc[:,:-5] - active_data_raman.iloc[:,:-5].mean(1).to_numpy()[:,None]) / active_data_raman.iloc[:,:-5].std(1).to_numpy()[:,None]


#ids_no_signal = np.concatenate([
#        ids_no_signal, 
#        np.array([
#            'MEC_04_014_DC_Active_20240808_S2_GridSi_20241210_91'
#        ])])

ids_no_signal = np.unique(
    np.concatenate([ids_no_signal,
    np.array(active_data_raman.loc[active_data_raman.iloc[:,:-5].T.std()<.01,'id'])]))

idx_raman_nan = np.unique(np.where(np.isnan(active_data_raman.iloc[:,:-5]))[0])


In [9]:
#active_data_raman['saturation'] = [x in ids_sat for x in active_data_raman['id']]
#active_data_raman['no_signal'] = [x in ids_no_signal for x in active_data_raman['id']]
#active_data_raman.to_csv(directory_name+'/active_data_raman_specy.csv')
#active_data_raman.to_csv(directory_name+'/active_data_raman_specy.csv')

In [10]:
remove_flopp = pd.read_excel('RochmanSpectra_ToRemove.xlsx', sheet_name="FLOPPe Remove")

flopp_df = pd.read_csv('processed_data/flopp_data_ir_final.csv')
del flopp_df['Unnamed: 0']
flopp_df.columns = list(np.float64(flopp_df.columns[:-4])) + list(flopp_df.columns[-4:])
flopp_df = flopp_df.query('polymer != "Cotton"').reset_index(drop=True)
flopp_df.columns = list(np.float64(flopp_df.columns[:-4])) + list(flopp_df.columns[-4:])
flopp_df = flopp_df.loc[[x not in np.array(remove_flopp['Example']) for x in flopp_df['id']], :].reset_index(drop=True)

rebecca_ir_df = pd.read_csv('processed_data/rebecca_data_ir_final.csv')
del rebecca_ir_df['Unnamed: 0']
rebecca_ir_df.columns = list(np.float64(rebecca_ir_df.columns[:-4])) + list(rebecca_ir_df.columns[-4:])

rebecca_ir_df = rebecca_ir_df.drop(np.where([x in list(mIRageLibraySpectra_Removed.query('Signal == "PTIR"')['ID']) for x in rebecca_ir_df['id']])[0],
                             axis=0).reset_index(drop=True)
rebecca_ir_df = rebecca_ir_df.query('polymer in @dict_slopp_reb').reset_index(drop=True)
rebecca_ir_df['polymer'] = [dict_slopp_reb[x] for x in rebecca_ir_df['polymer']]

In [11]:
folders = np.array(os.listdir('processed_data'))
file_names = folders[['active_data' in x and 'ir.' in x for x in folders]]
file_names = ['processed_data/'+x for x in file_names]
active_data_ir_list = []

for f in file_names:
    if f == 'active_data_all_ir.csv':
        continue
    active_data_ir = pd.read_csv(f)
    del active_data_ir['Unnamed: 0']
    active_data_ir.columns = list(np.float64(active_data_ir.columns[:-4])) + list(active_data_ir.columns[-4:])

    active_data_ir['polymer'] = ''
    active_data_ir['label'] = ''

    active_data_ir_list.append(active_data_ir)
    
active_data_ir = pd.concat(active_data_ir_list, axis=0).reset_index(drop=True)
background_ids = np.array(active_data_ir.query('background == "background"')['id'])

flopp_df.iloc[:,:-4] = (flopp_df.iloc[:,:-4] - flopp_df.iloc[:,:-4].mean(1).to_numpy()[:,None]) / flopp_df.iloc[:,:-4].std(1).to_numpy()[:,None]
rebecca_ir_df.iloc[:,:-4] = (rebecca_ir_df.iloc[:,:-4] - rebecca_ir_df.iloc[:,:-4].mean(1).to_numpy()[:,None]) / rebecca_ir_df.iloc[:,:-4].std(1).to_numpy()[:,None]
active_data_ir.iloc[:,:-6] = (active_data_ir.iloc[:,:-6] - active_data_ir.iloc[:,:-6].mean(1).to_numpy()[:,None]) / active_data_ir.iloc[:,:-6].std(1).to_numpy()[:,None]


In [12]:
#active_data_ir.to_csv(directory_name+'/active_data_ir.csv')

In [13]:
ids_rubber = pd.read_excel('evaluation_data/Labeled_D&G_RamanOnly_Passive.xlsx', header=None)[0].to_numpy()
rubber_data = active_data_raman.query('id in @ids_rubber').reset_index(drop=True)#.iloc[:,:-5]
rubber_data['polymer'] = 'Rubber'
rubber_data = rubber_data.drop('label',axis=1)
rubber_data = rubber_data.iloc[:,list(range(rubber_data.shape[1]-2))+[-1,-2]]
active_data_raman_rest = active_data_raman.query('id not in @ids_rubber').reset_index(drop=True)

In [14]:
rubber_data =  pd.read_csv(directory_name+"/passive_uncertain_rubber_data_raman_specy.csv")
del rubber_data["Unnamed: 0"]
rubber_data

,602.0,604.0,606.0,608.0,610.0,612.0,614.0,616.0,618.0,620.0,...,3488.0,3490.0,3492.0,3494.0,3496.0,3498.0,map_x,map_y,polymer,id
0,-0.666961,-0.405833,-0.019063,0.401075,0.848068,1.043228,1.177303,1.254237,1.293004,1.259763,...,-0.882203,-0.882203,-0.882203,-0.882203,-0.882203,-0.882203,16653.849609,784.150024,Rubber,FHS_passive_July2024_D6_20240807_113
1,-0.932589,-0.887620,-0.732531,-0.606890,-0.504951,-0.215130,0.104644,0.292878,0.423798,0.446588,...,-0.932589,-0.932589,-0.932589,-0.932589,-0.932589,-0.932589,20719.550781,-908.599976,Rubber,DC_passive_July2024_I9_20240806_64
2,0.989372,1.191921,1.298724,1.457655,1.718501,1.914340,2.067306,2.133449,2.161686,2.026951,...,-0.797338,-0.797338,-0.797338,-0.797338,-0.797338,-0.797338,12706.849609,-9468.099609,Rubber,ICB_passive_November2024_C10_20241126_28
3,-0.632852,-0.562823,-0.541976,-0.528556,-0.529652,-0.535728,-0.545088,-0.619458,-0.692770,-0.692770,...,-0.692770,-0.692770,-0.692770,-0.692770,-0.692770,-0.692770,18742.750000,-3899.949951,Rubber,DC_passive_November2024_C5_20241119_4
4,0.325980,0.788182,0.995317,1.140202,1.163383,1.102225,0.985445,0.842060,0.668463,0.321542,...,-0.920349,-0.920349,-0.920349,-0.920349,-0.920349,-0.920349,14133.549805,-4123.000000,Rubber,DC_passive_November2024_C5_20241113_43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,-0.768037,-0.750531,-0.514626,-0.281641,-0.054364,0.127710,0.279974,0.335705,0.365438,0.321123,...,-0.768037,-0.768037,-0.768037,-0.768037,-0.768037,-0.768037,15578.049805,-1201.250000,Rubber,DC_passive_MarApr2024_C9_20240612_61
107,-0.629589,-0.506005,-0.313060,-0.156681,-0.071791,-0.023516,0.000612,-0.045187,-0.095287,-0.102616,...,-0.725190,-0.725190,-0.725190,-0.725190,-0.725190,-0.725190,15207.799805,-1024.500000,Rubber,DC_passive_MarApr2024_C9_20240612_95
108,-0.631786,-0.631786,-0.631786,-0.583302,-0.440028,-0.340757,-0.270506,-0.252030,-0.250739,-0.310671,...,-0.631786,-0.631786,-0.631786,-0.631786,-0.631786,-0.631786,15443.400391,-1040.150024,Rubber,DC_passive_MarApr2024_C9_20240612_144
109,-0.695567,-0.695567,-0.695567,-0.628524,-0.430405,-0.320419,-0.268555,-0.280863,-0.302828,-0.323432,...,-0.695567,-0.695567,-0.695567,-0.695567,-0.695567,-0.695567,16020.650391,-1187.050049,Rubber,DC_passive_MarApr2024_C9_20240612_172


In [15]:
#active_data_ir_paired = active_data_ir.query('id not in @ids_sat and id not in @background_ids and id not in @ids_no_signal').reset_index(drop=True)
#active_data_ir_unpaired_ram = active_data_ir.query('(id not in @ids_sat and id not in @ids_no_signal) and id in @background_ids').reset_index(drop=True)
#active_data_ir_unpaired_ir = active_data_ir.query('(id in @ids_sat or id in @ids_no_signal) and id not in @background_ids').reset_index(drop=True)
#active_data_ir_nopair = active_data_ir.query('(id in @ids_sat or id in @ids_no_signal) and id in @background_ids').reset_index(drop=True)

active_data_raman_paired = active_data_raman.query('(id not in @ids_sat and id not in @ids_no_signal) and id not in @background_ids').reset_index(drop=True)
active_data_raman_unpaired_ram = active_data_raman.query('(id not in @ids_sat and id not in @ids_no_signal) and id in @background_ids').reset_index(drop=True)
active_data_raman_unpaired_ir = active_data_raman.query('(id in @ids_sat or id in @ids_no_signal) and id not in @background_ids').reset_index(drop=True)
active_data_raman_nopair = active_data_raman.query('(id in @ids_sat or id in @ids_no_signal) and id in @background_ids').reset_index(drop=True)

In [16]:
print('unpaired Raman spectra: ', active_data_raman_unpaired_ram.shape[0])
print('unpaired IR spectra: ', active_data_raman_unpaired_ir.shape[0])
print('no pair spectra: ', active_data_raman_nopair.shape[0])
print('paired spectra: ', active_data_raman_paired.shape[0])

unpaired Raman spectra:  491
unpaired IR spectra:  1401
no pair spectra:  65
paired spectra:  1364


In [17]:
np.random.seed(0)
idx = np.random.rand(rubber_data.shape[0]) > .50
rubber_ref = rubber_data.loc[idx,:].reset_index(drop=True)
rubber_cal = rubber_data.loc[~idx,:].reset_index(drop=True)

In [18]:
active_data_raman_all = pd.concat([active_data_raman_paired,
                                active_data_raman_unpaired_ram,
                                active_data_raman_unpaired_ir,
                                active_data_raman_nopair], axis=0).reset_index(drop=True)
active_data_raman_all['pair'] = (
    ['paired']*active_data_raman_paired.shape[0]+
    ['unpaired_ram']*active_data_raman_unpaired_ram.shape[0]+
    ['unpaired_ir']*active_data_raman_unpaired_ir.shape[0]+
    ['nopair']*active_data_raman_nopair.shape[0]
)
#active_data_raman_all.to_csv('active_data_all_raman.csv')

In [19]:
flopp_slopp_labels = np.array(slopp_df.groupby('polymer').count().iloc[:,-1:].merge(flopp_df.groupby('polymer').count().iloc[:,-1:], 
                                                      left_index=True,
                                                      right_index=True, how='inner').index)

flopp_df = flopp_df.query('polymer in @flopp_slopp_labels').reset_index(drop=True)
slopp_df = slopp_df.query('polymer in @flopp_slopp_labels').reset_index(drop=True)


In [20]:
idx_both_rebecca = np.array(rebecca_ir_df.loc[[x in np.array(rebecca_raman_df['id']) for x in rebecca_ir_df['id']],'id'])# 
rebecca_ir_df = rebecca_ir_df.query('id in @idx_both_rebecca').reset_index(drop=True)
rebecca_raman_df = rebecca_raman_df.query('id in @idx_both_rebecca').reset_index(drop=True)


# Paired

In [40]:

(reference_library_raman, 
 calibration_set_raman, 
 test_set_raman, 
 label_table_raman) = gen_data_sets(rubber_ref, 
                              rubber_cal, 
                              active_data_raman_paired)

(X_train_raman, X_cal_raman, X_test_raman, 
 y_train_raman, y_cal_raman, y_test_raman, 
 label_train_raman, label_cal_raman, label_test_raman, 
 n_test_raman, wl_list_raman) = process_data(reference_library_raman,
                                             calibration_set_raman,
                                             test_set_raman)

In [41]:
def smx_log(X_train, X_test, labels_train, C=0.01, return_weights=False):
    
    clf = LogisticRegression(random_state=0, C=C).fit(X_train, labels_train)
    smx = clf.predict_proba(X_test)
    label_pred = clf.predict(X_test)
    if return_weights:
        return smx, clf
    
    return smx

In [45]:
alpha = .1
# Calculate HQI for calibration set
smx_cal_raman = smx_pcc(X_train_raman, 
                  X_cal_raman, 
                  label_train_raman, 
                        #C=0.002
                       )

# Calculate HQI for test set
smx_test_raman = smx_pcc(X_train_raman, 
                   X_test_raman, 
                   label_train_raman, 
                               # C=0.002
                               )

#smx_test_all = smx_nn_all(
#        X_train_raman, 
#        X_test_raman, 
#        label_train_raman
#)

#pred_slopp_df = [
#    np.array(label_table_raman.iloc[np.argmax(smx_test, 1),1]),
#    np.max(smx_test, 1),
#    np.array(reference_library_raman.iloc[np.argmax(smx_test_all, 0),0])
#]

# Set confidence level 1 - alpha
n = smx_cal_raman.shape[0]
#for alpha in [alpha]:
prediction_sets_raman, qhat_raman = conform_pred(smx_cal_raman, smx_test_raman, label_cal_raman, n, alpha, cond=0)

prediction_sets_n_raman = [list(label_table_raman.loc[ps,'polymer']) for ps in prediction_sets_raman]
prediction_sets_size_raman = np.array([len(ps) for ps in prediction_sets_n_raman])

zero_label_raman = np.where(prediction_sets_size_raman == 0)[0]
single_label_raman = np.where(prediction_sets_size_raman == 1)[0]
mult_label_raman = np.where(prediction_sets_size_raman > 1)[0]

print('Sets with zero labels: ' + str(zero_label_raman.shape[0]))
print('Sets with one label: ' + str(single_label_raman.shape[0]))
print('Sets with >1 labels: ' + str(mult_label_raman.shape[0]))
print('Mean set size:', prediction_sets_raman.sum(1).mean())
print('Std set size:', prediction_sets_raman.sum(1).std())
print('HQI cutt off:', 1-qhat_raman)

Sets with zero labels: 973
Sets with one label: 391
Sets with >1 labels: 0
Mean set size: 0.2866568914956012
Std set size: 0.4521998651632709
HQI cutt off: 0.9577787447942816


In [47]:
output = [prediction_sets_n_raman[i][0] if i in single_label_raman else 
              '--outlier--' if i in zero_label_raman else 
              '--manual check!--' for i in range(label_test_raman.shape[0])]
output_df_raman = pd.DataFrame({'id':test_set_raman['id'],
                          'prection_set_raman':prediction_sets_n_raman,
                              #'output':output
                              })
output_df_raman.to_csv(directory_name+'/CP_Rubber_results_active_paired.csv')
output_df_raman

,id,prection_set_raman
0,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,[]
1,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,[]
2,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,[]
3,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,[]
4,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,[]
...,...,...
1359,MEC_04_014_DC_Active_20240808_S1_GridSi_202503...,[]
1360,MEC_04_014_DC_Active_20240808_S1_GridSi_202503...,[]
1361,MEC_04_014_DC_Active_20240808_S1_GridSi_202503...,[]
1362,MEC_04_014_DC_Active_20240808_S1_GridSi_202503...,[]


# Unpaired Raman

In [21]:
(reference_library_raman, 
 calibration_set_raman, 
 test_set_raman, 
 label_table_raman) = gen_data_sets(rubber_ref, 
                              rubber_cal, 
                              active_data_raman_unpaired_ram)

(X_train_raman, X_cal_raman, X_test_raman, 
 y_train_raman, y_cal_raman, y_test_raman, 
 label_train_raman, label_cal_raman, label_test_raman, 
 n_test_raman, wl_list_raman) = process_data(reference_library_raman,
                                 calibration_set_raman,
                                 test_set_raman)


In [23]:
# Calculate HQI for calibration set
smx_cal = smx_pcc(X_train_raman, 
                 X_cal_raman, 
                 label_train_raman, 
                 # C=.002
                 )

# Calculate HQI for test set
smx_test = smx_pcc(X_train_raman, 
                  X_test_raman, 
                  label_train_raman, 
                  # C=.002
                  )

#smx_test_all = smx_nn_all(
#        X_train_raman, 
#        X_test_raman, 
#        label_train_raman
#)

#pred_slopp_df = [
#    np.array(label_table_raman.iloc[np.argmax(smx_test, 1),1]),
#    np.max(smx_test, 1),
#    np.array(reference_library_raman.iloc[np.argmax(smx_test_all, 0),0])
#]

# Set confidence level 1 - alpha
n = smx_cal.shape[0]
alpha = 0.1
#for alpha in [alpha]:
prediction_sets_raman, qhat_raman = conform_pred(smx_cal, smx_test, label_cal_raman, n, alpha, cond=0)

prediction_sets_n_raman = [list(label_table_raman.loc[ps,'polymer']) for ps in prediction_sets_raman]
prediction_sets_size_raman = np.array([len(ps) for ps in prediction_sets_n_raman])

zero_label_raman = np.where(prediction_sets_size_raman == 0)[0]
single_label_raman = np.where(prediction_sets_size_raman == 1)[0]
mult_label_raman = np.where(prediction_sets_size > 1)[0]

print('Sets with zero labels: ' + str(zero_label_raman.shape[0]))
print('Sets with one label: ' + str(single_label_raman.shape[0]))
print('Sets with >1 labels: ' + str(mult_label_raman.shape[0]))
print('Mean set size:', prediction_sets_raman.sum(1).mean())
print('Std set size:', prediction_sets_raman.sum(1).std())
print('HQI cutt off:', 1-qhat_raman)

Sets with zero labels: 299
Sets with one label: 192
Sets with >1 labels: 0
Mean set size: 0.3910386965376782
Std set size: 0.4879830267005112
HQI cutt off: 0.9577787447942816


In [25]:
output = [prediction_sets_n_raman[i][0] if i in single_label_raman else 
              '--outlier--' if i in zero_label_raman else 
              '--manual check!--' for i in range(label_test_raman.shape[0])]
output_df_raman = pd.DataFrame({'id':test_set_raman['id'],
                          'prection_set_raman':prediction_sets_n_raman,
                              #'output':output
                              })
output_df_raman.to_csv(directory_name+'/CP_Rubber_results_active_unpaired.csv')
output_df_raman

,id,prection_set_raman
0,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,[]
1,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,[Rubber]
2,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,[]
3,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,[Rubber]
4,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,[]
...,...,...
486,MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...,[Rubber]
487,MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...,[]
488,MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...,[]
489,MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...,[]
